In [2]:
import pandas as pd
import numpy as np

In [12]:
# load the provided data
train_features = pd.read_csv(
    "../data/dengue_features_train.csv", index_col=[0, 1, 2]
)

train_labels = pd.read_csv(
    "../data/dengue_labels_train.csv", index_col=[0, 1, 2]
)

sj_train_features = train_features.loc["sj"]
sj_train_labels = train_labels.loc["sj"]

# Separate data for Iquitos
iq_train_features = train_features.loc["iq"]
iq_train_labels = train_labels.loc["iq"]
sj_train_features["total_cases"] = sj_train_labels.total_cases
iq_train_features["total_cases"] = iq_train_labels.total_cases

In [11]:
sj_train_subtrain = sj_train_features.head(800)
sj_train_subtest = sj_train_features.tail(sj_train_features.shape[0] - 800)

iq_train_subtrain = iq_train_features.head(400)
iq_train_subtest = iq_train_features.tail(iq_train_features.shape[0] - 400)

sj_train_features["total_cases"] = sj_train_labels.total_cases
iq_train_features["total_cases"] = iq_train_labels.total_cases

In [13]:
from statsmodels.tools import eval_measures
import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split
import statsmodels.api as sm

def get_best_model(train, test):
    # Step 1: specify the form of the model
    model_formula = (
        "total_cases ~ 1 + "
        "reanalysis_specific_humidity_g_per_kg + "
        "reanalysis_dew_point_temp_k + "
        "station_min_temp_c + "
        "station_avg_temp_c"
    )

    grid = 10 ** np.arange(-8, -3, dtype=np.float64)

    best_alpha = []
    best_score = 1000

    # Step 2: Find the best hyper parameter, alpha
    for alpha in grid:
        model = smf.glm(
            formula=model_formula,
            data=train,
            family=sm.families.NegativeBinomial(alpha=alpha),
        )

        results = model.fit()
        predictions = results.predict(test).astype(int)
        score = eval_measures.meanabs(predictions, test.total_cases)

        if score < best_score:
            best_alpha = alpha
            best_score = score

    print("best alpha = ", best_alpha)
    print("best score = ", best_score)

    # Step 3: refit on entire dataset
    full_dataset = pd.concat([train, test])
    model = smf.glm(
        formula=model_formula,
        data=full_dataset,
        family=sm.families.NegativeBinomial(alpha=best_alpha),
    )

    fitted_model = model.fit()
    return fitted_model


sj_best_model = get_best_model(sj_train_subtrain, sj_train_subtest)
iq_best_model = get_best_model(iq_train_subtrain, iq_train_subtest)

best alpha =  1e-08
best score =  21.926470588235293


IntCastingNaNError: Cannot convert non-finite values (NA or inf) to integer.Replace or remove non-finite values or cast to an integer typethat supports these values (e.g. 'Int64')